In [1]:
%%writefile /media/notebooks/EjercicioFinal/mapreduce_csv.sh
#!/bin/bash

# Directorio de salida
OUTPUT="/datasets/synthea/join_clinico_csv"

# Eliminamos la salida anterior si existe
hadoop fs -rm -r -f "$OUTPUT"

# Ejecución de Hadoop Streaming
hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -files mapper_csv.py,reduce_csv.py \
    -input /datasets/synthea/csv/patients.csv \
    -input /datasets/synthea/csv/encounters.csv \
    -input /datasets/synthea/csv/organizations.csv \
    -output "$OUTPUT" \
    -mapper "python3 mapper_csv.py" \
    -reducer "python3 reduce_csv.py" 
    

Overwriting /media/notebooks/EjercicioFinal/mapreduce_csv.sh


In [2]:


%%writefile /media/notebooks/EjercicioFinal/mapreduce_parquet.sh
#!/bin/bash

OUTPUT="/datasets/synthea/join_clinico_parquet"

echo "Iniciando MapReduce sobre Parquet..."

# Eliminar salida anterior
hadoop fs -rm -r -f "$OUTPUT"

# Ejecutar Hadoop Streaming
hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -input /datasets/synthea/parquet/patients.parquet \
    -input /datasets/synthea/parquet/encounters.parquet \
    -input /datasets/synthea/parquet/organizations.parquet \
    -output "$OUTPUT" \
    -mapper "python3 mapper_parquet.py" \
    -reducer "python3 reduce_parquet.py" \
    -files mapper_parquet.py,reduce_parquet.py

echo "MapReduce sobre Parquet finalizado."



Overwriting /media/notebooks/EjercicioFinal/mapreduce_parquet.sh


In [3]:
%%writefile /media/notebooks/EjercicioFinal/mapreduce_avro.sh
#!/bin/bash

OUTPUT="/datasets/synthea/join_clinico_avro"

echo "Iniciando MapReduce sobre Avro..."

# Eliminar salida anterior
hadoop fs -rm -r -f "$OUTPUT"

# Ejecutar Hadoop Streaming
hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -input /datasets/synthea/avro/patients.avro \
    -input /datasets/synthea/avro/encounters.avro \
    -input /datasets/synthea/avro/organizations.avro \
    -output "$OUTPUT" \
    -mapper "python3 mapper_avro.py" \
    -reducer "python3 reduce_avro.py" \
    -files mapper_avro.py,reduce_avro.py

echo "MapReduce sobre Avro finalizado."



Overwriting /media/notebooks/EjercicioFinal/mapreduce_avro.sh


In [4]:
%%writefile /media/notebooks/EjercicioFinal/mapreduce_avro.sh
#!/bin/bash

OUTPUT="/datasets/synthea/join_clinico_avro"

echo "Iniciando MapReduce sobre Avro..."

# Eliminar salida anterior
hadoop fs -rm -r -f "$OUTPUT"

# Ejecutar Hadoop Streaming
hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -input /datasets/synthea/avro \
    -input /datasets/synthea/avro/encounters \
    -input /datasets/synthea/avro/organizations \
    -output "$OUTPUT" \
    -mapper "python3 mapper_avro.py" \
    -reducer "python3 reduce_avro.py" \
    -files mapper_avro.py,reduce_avro.py

echo "MapReduce sobre Avro finalizado."

Overwriting /media/notebooks/EjercicioFinal/mapreduce_avro.sh


In [5]:
%%writefile /media/notebooks/EjercicioFinal/hive_csv.sh

hive -f /media/notebooks/EjercicioFinal/join_csv.hql

echo "Csv-Proceso finalizado."



Overwriting /media/notebooks/EjercicioFinal/hive_csv.sh


In [6]:
%%writefile /media/notebooks/EjercicioFinal/hive_parquet.sh
#!/bin/bash

hive -f /media/notebooks/EjercicioFinal/join_clinico_parquet.hql

echo "Parquet-Proceso finalizado."

Overwriting /media/notebooks/EjercicioFinal/hive_parquet.sh


In [7]:
%%writefile /media/notebooks/EjercicioFinal/hive_avro.sh
#!/bin/bash

hive -f /media/notebooks/EjercicioFinal/join_clinico_avro.hql

echo "Avro-Proceso finalizado."

Overwriting /media/notebooks/EjercicioFinal/hive_avro.sh


In [8]:
%%writefile /media/notebooks/EjercicioFinal/MonitorProceso.py
import subprocess
import time
import psutil


class MonitorProceso:

    def __init__(self, script):
        self.script = script
        self.tiempo = 0
        self.memoria_maxima = 0

    def ejecutar(self):

        inicio = time.perf_counter()

        proceso = subprocess.Popen(
            ["bash", self.script],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE
        )

        proceso_psutil = psutil.Process(proceso.pid)

        while proceso.poll() is None:

            try:
                memoria_actual = (
                    proceso_psutil.memory_info().rss
                    / (1024 * 1024)
                )

                self.memoria_maxima = max(
                    self.memoria_maxima,
                    memoria_actual
                )

            except psutil.NoSuchProcess:
                break

            time.sleep(0.5)

        fin = time.perf_counter()

        self.tiempo = fin - inicio

        return {
            "script": self.script,
            "tiempo": self.tiempo,
            "memoria": self.memoria_maxima,
            "codigo_salida": proceso.returncode
        }

Overwriting /media/notebooks/EjercicioFinal/MonitorProceso.py


In [9]:
%%writefile /media/notebooks/EjercicioFinal/medir_procesos.py
from MonitorProceso import MonitorProceso


procesos = [
    ("MapReduce CSV", "mapreduce_csv.sh"),
    ("MapReduce Parquet", "mapreduce_parquet.sh"),
    ("MapReduce Avro", "mapreduce_avro.sh"),
    ("Hive CSV", "hive_csv.sh"),
    ("Hive Parquet", "hive_parquet.sh"),
    ("Hive Avro", "hive_avro.sh")
]


resultados = []


for nombre, script in procesos:

    print(f"\nEjecutando: {nombre}")

    monitor = MonitorProceso(script)

    resultado = monitor.ejecutar()

    resultados.append({
        "proceso": nombre,
        "tiempo": resultado["tiempo"],
        "memoria": resultado["memoria"],
        "codigo_salida": resultado["codigo_salida"]
    })

    print(
        f"Tiempo de ejecución: "
        f"{resultado['tiempo']:.2f} segundos"
    )

    print(
        f"Memoria máxima: "
        f"{resultado['memoria']:.2f} MB"
    )

    print(
        f"Código de salida: "
        f"{resultado['codigo_salida']}"
    )

Overwriting /media/notebooks/EjercicioFinal/medir_procesos.py


In [10]:
cd EjercicioFinal/

/media/notebooks/EjercicioFinal


/usr/local/lib/python3.10/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [ ]:
!python3 /media/notebooks/EjercicioFinal/medir_procesos.py


Ejecutando: MapReduce CSV
Tiempo de ejecución: 122.35 segundos
Memoria máxima: 3.30 MB
Código de salida: 0

Ejecutando: MapReduce Parquet
Tiempo de ejecución: 4.05 segundos
Memoria máxima: 3.34 MB
Código de salida: 0

Ejecutando: MapReduce Avro
Tiempo de ejecución: 4.52 segundos
Memoria máxima: 3.31 MB
Código de salida: 0

Ejecutando: Hive CSV
Tiempo de ejecución: 54.12 segundos
Memoria máxima: 3.17 MB
Código de salida: 0

Ejecutando: Hive Parquet
Tiempo de ejecución: 140.79 segundos
Memoria máxima: 3.32 MB
Código de salida: 0

Ejecutando: Hive Avro
